# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print dataset name and description (access as attributes)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields. Here we enumerate all record sets, then all fields and columns in each record set.

In [ ]:
# List all Record Sets in the dataset (by @id)
print('Available Record Sets:')
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"  [@id] {rs.id} | name: {rs.name}")

# For each Record Set, list fields (by @id) and linked columns (by @id and name)
for rs in record_sets:
    print(f"\nRecord Set: {rs.name} ([@id] {rs.id})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    [@id] {field.id} | name: {field.name}")
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"      -> Column [@id] {col.id} | name: {col.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview (see above).

In [ ]:
# Choose which record set(s) to extract (referenced by their @id)
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_sets_ids:
    try:
        # Records yields dicts with field @id as keys
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from Record Set [@id] {record_set_id}")
    except Exception as e:
        print(f"Failed to load records for [@id] {record_set_id}: {e}")

# Show available columns for the first record set (or change as relevant)
if record_sets_ids:
    first_rs_id = record_sets_ids[0]
    print(f"\nColumns in record set [@id] {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping by key attributes. All data columns are referenced by their `@id`.

In [ ]:
# Pick a record set and look for a numeric field
record_set_id = record_sets_ids[0]  # Use first or change as needed
df = dataframes[record_set_id]

print(f"Sample columns in [{record_set_id}] DataFrame:")
print(df.columns.tolist())

# Identify a likely numeric field (by @id, e.g., 'cr:MW' for molecular weight, etc.)
numeric_field_candidates = [col for col in df.columns if 'MW' in col or 'abundance' in col or 'count' in col or 'coverage' in col or 'peptide' in col]
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]  # choose first one
else:
    numeric_field_id = df.columns[0]  # fallback
print(f"Using numeric field [@id] {numeric_field_id}")

# Filter rows where [numeric_field] > threshold
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10 # fallback
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold]
print(f"Filtered to {len(filtered_df)} records with {numeric_field_id} > {threshold}")

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"],].head())

# Group by a categorical field, if available (use @id)
group_field_candidates = [col for col in df.columns if 'description' in col or 'accession' in col or 'group' in col or 'sample' in col]
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field:
    print(f"Grouping by field [@id] {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(grouped_df.head())

## 5. Visualization
Visualize the data distribution or relationships for fields of interest. Here we'll plot the distribution of the selected numeric field and values grouped by the selected category (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field
plt.figure(figsize=(6,4))
sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouping field exists, plot group means
if group_field:
    plt.figure(figsize=(8,4))
    group_means = df.groupby(group_field)[numeric_field_id].mean().sort_values()
    group_means.plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We explored the FAIR² mass spectrometry dataset—loaded via its Croissant schema and processed with `mlcroissant`. After reviewing its structure via `@id` references for record sets and fields, we extracted data, performed basic filtering and normalization by field, and visualized the results.

- The Croissant schema enables easy discovery of the dataset's organization (fields, columns, record sets).
- EDA was demonstrated referencing dataset elements strictly by their `@id` for reproducibility.
- You can continue refining analysis by selecting and processing further fields based on scientific or machine learning needs.

Explore the full documentation for [mlcroissant](https://github.com/mlcommons/croissant) for advanced usage!